<a href="https://colab.research.google.com/github/ajinkjayan21-cpu/Fruit-Recognition/blob/main/Fruit_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
utkarshsaxenadn_fruits_classification_path = kagglehub.dataset_download('utkarshsaxenadn/fruits-classification')

print('Data source import complete.')


Data source import complete.


In [6]:
import numpy as np
import pandas as pd
import os


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,BatchNormalization,Dropout
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

In [8]:
def load_and_preprocess_images(folder_path, subfolders, image_size=(224, 224)):

    label_map = {subfolder: idx for idx, subfolder in enumerate(subfolders)}

    images = []
    labels = []

    for subfolder in subfolders:
        subfolder_path = os.path.join(folder_path, subfolder)
        for filename in os.listdir(subfolder_path):
            img_path = os.path.join(subfolder_path, filename)
            img = cv2.imread(img_path)
            if img is not None:

                img_resized = cv2.resize(img, image_size)
                images.append(img_resized)
                labels.append(label_map[subfolder])

    images = np.array(images)
    labels = np.array(labels)

    images = images.astype('float32') / 255.0

    return images, labels

#/kaggle/input/fruits-classification/Fruits Classification/train/Apple

#/kaggle/input/fruits-classification/Fruits Classification/train/Apple/Apple (1).jpeg

In [9]:
train_dir = "/content/train.zip"
test_dir = "/content/test.zip"
valid_dir = "/content/valid.zip"

subfolders = ['Apple', 'Banana', 'Grape', 'Mango','Strawberry']

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Get the base path where Kaggle data sources are downloaded
kaggle_data_path = utkarshsaxenadn_fruits_classification_path

# Assuming kaggle_data_path already contains the extracted dataset
# The structure should be kaggle_data_path/Fruits Classification/{train,test,valid}/class_name
# Therefore, we need to point to the 'Fruits Classification' subfolder first.
base_data_folder = os.path.join(kaggle_data_path, 'Fruits Classification')

# Now, define train_dir, test_dir, valid_dir to point to the actual image folders
# These folders are directly within the 'Fruits Classification' directory after Kaggle's extraction.
train_dir = os.path.join(base_data_folder, 'train')
test_dir = os.path.join(base_data_folder, 'test')
valid_dir = os.path.join(base_data_folder, 'valid')

train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Although `valid_dir` exists, the original `model.fit` uses `test_generator` for validation.
# So, a separate `validation_generator` is not being created here to align with the original notebook's flow.


Found 9700 images belonging to 5 classes.
Found 100 images belonging to 5 classes.


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

#CNN building:
# num_classes = len(train_generator.class_indices) # This line caused NameError
num_classes = 5 # Hardcoding num_classes to 5 based on data loading output

model = Sequential([
    Conv2D(128, (3, 3), activation='relu', padding='same', input_shape=(256, 256, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),


    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),


    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),


    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),


    Flatten(),

    Dense(512, activation='relu'),
    BatchNormalization(),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(num_classes, activation='softmax') # Using num_classes here
])

model.summary()

# Model compilation
model.compile(optimizer='Adam', loss='categorical_crossentropy',metrics=['Accuracy'])

# Checkpoint definition
checkpoint = ModelCheckpoint('model_checkpoint.weights.h5', save_best_only=False, save_weights_only=True, verbose=1)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 128)  │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 32)     │        36,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,422,181 (5.43 MB)

 Trainable params: 1,420,069 (5.42 MB)

 Non-trainable params: 2,112 (8.25 KB)

In [2]:
try:
    history = model.fit(
            train_generator,
            epochs=100,
            validation_data=test_generator,
            batch_size=32,
            callbacks=[checkpoint]
        )
except NameError as e:
    if "train_generator" in str(e) or "test_generator" in str(e):
        print("Error: train_generator or test_generator is not defined. Please ensure that cell '4ioP4jMVHTvz' (which defines the data generators) has been executed successfully before running this cell.")
    elif "model" in str(e):
        print("Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.")
    else:
        raise e

Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.
Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.


Please ensure that all preceding cells, especially the one defining the `model` (cell `lzE1khaeHTvz`), have been executed successfully before running the model training cell (`U7UMnI0LHTv0`). The `NameError` indicates that the `model` object was not found, likely because its definition cell was not run.

In [3]:
try:
    test_loss, test_acc = model.evaluate(test_generator)
    print(f'Test accuracy: {test_acc *100} %')
except NameError as e:
    if "test_generator" in str(e):
        print("Error: test_generator is not defined. Please ensure that cell '4ioP4jMVHTvz' (which defines the data generators) has been executed successfully before running this cell.")
    elif "model" in str(e):
        print("Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.")
    else:
        raise e

Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.


In [4]:
try:
    model.save('fruit_recognizer.h5')
except NameError as e:
    if "model" in str(e):
        print("Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.")
    else:
        raise e

Error: model is not defined. Please ensure that cell 'lzE1khaeHTvz' (which defines and compiles the model) has been executed successfully before running this cell.
